# Single-stage optimization checks

Interactive equivalents of `test_optimize.py`. MMFF and UFF use RDKit locally, so no optional executable is required.

In [1]:
from pathlib import Path
import sys
import numpy as np
from rdkit import Chem

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))

from ensemblelab import generate
from ensemblelab.optimizers.mmff import MMFFOptimizer, UFFOptimizer
from ensemblelab.optimizers.xtb import GFN2xTBOptimizer


## MMFF optimization contract

`MMFFOptimizer` returns a new ensemble, stores energies in kcal/mol, records convergence and provenance, and keeps RDKit and ASE coordinates synchronized.

In [2]:
ensemble = generate('CCO', n_confs=2)
original_positions = [conformer.atoms.positions.copy() for conformer in ensemble.conformers]
optimized = MMFFOptimizer(max_steps=100).optimize(ensemble)

assert optimized is not ensemble
assert all(conformer.energy is None for conformer in ensemble.conformers)
assert all(conformer.energy is not None for conformer in optimized.conformers)
assert all(conformer.energy_unit == 'kcal/mol' for conformer in optimized.conformers)
assert all(conformer.optimization_method == 'MMFF' for conformer in optimized.conformers)
assert all(conformer.optimization_converged is True for conformer in optimized.conformers)
assert optimized.metadata['energy_status'] == 'computed'
assert optimized.metadata['optimization_status'] == 'optimized'
assert optimized.metadata['history'][-1]['process'] == 'optimization'
assert optimized.metadata['history'][-1]['method'] == 'MMFF'

for before, original, result in zip(original_positions, ensemble.conformers, optimized.conformers, strict=True):
    np.testing.assert_allclose(original.atoms.positions, before)
    np.testing.assert_allclose(result.atoms.positions, optimized.rdkit_conformer(result.id).GetPositions())
    assert isinstance(result._molecule, Chem.Mol)
    assert result._molecule is optimized.molecule

print('MMFF contract passed.')
[(conformer.id, conformer.energy) for conformer in optimized.conformers]

MMFF contract passed.


[(0, -1.3368570639146007), (1, -1.3368570626158607)]

## UFF optimizer and optional xTB backend

`UFFOptimizer` shares the same result contract. GFN2-xTB is optional and should either optimize successfully when TBLite is installed or raise the documented dependency error.

In [3]:
uff_ensemble = UFFOptimizer(max_steps=100).optimize(ensemble)
assert uff_ensemble is not ensemble
assert all(conformer.energy is not None for conformer in uff_ensemble.conformers)
assert all(conformer.energy_unit == 'kcal/mol' for conformer in uff_ensemble.conformers)
assert all(conformer.optimization_method == 'UFF' for conformer in uff_ensemble.conformers)
assert uff_ensemble.metadata['history'][-1]['method'] == 'UFF'

try:
    xtb_ensemble = GFN2xTBOptimizer(max_steps=1).optimize(ensemble)
except ImportError as error:
    assert "requires the optional 'tblite' package" in str(error)
    print(f'Expected optional-backend error: {error}')
else:
    assert all(conformer.energy is not None for conformer in xtb_ensemble.conformers)
    assert xtb_ensemble.metadata['history'][-1]['method'] == 'GFN2-xTB'
    print('GFN2-xTB contract passed.')

print('UFF contract passed.')

TypeError: super(type, obj): obj (instance of GFN2xTBOptimizer) is not an instance or subtype of type (GFN2xTBOptimizer).